# Lab — DQN with Experience Replay and Target Network

**Intelligent Agents · UC2 · §2.2**

In this lab you build a Deep Q-Network (DQN) in PyTorch that solves **CartPole-v1**. Then you run experiments that show what experience replay and the target network each contribute. The code follows the algorithm from the reading (Hu, 2023), step by step.

**By the end you should be able to:**

- Implement a uniform ring-buffer replay memory with `add()` and `sample()`.
- Build a Q-network as a multilayer perceptron, plus a frozen target copy θ⁻.
- Compute TD targets that handle terminal states correctly, and train with mini-batch SGD.
- Explain, using your own plots, why replay and the target network make learning stable.

**Prerequisites:** Python, NumPy basics, PyTorch tensors and autograd (see the MLP tutorial), and the tabular Q-learning update.

**Format:** pairs, two 2-hour sessions — Parts 1–3 in session 1, Parts 4–5 in session 2.

**How to work in this notebook**

- Complete every cell marked `# TODO`. Replace each `raise NotImplementedError` with your code.
- Run the **self-check** cell after each part. Do not move on until it prints *all checks passed*.
- Do not edit the self-check cells.
- Write your answers to the questions in the **Your answer** cells.

## Setup

You need Python 3.10+, PyTorch and Gymnasium. Everything runs on a laptop CPU; no GPU is needed.

**Environment — CartPole-v1.** A pole is hinged on a cart. The agent pushes the cart left or right to keep the pole upright.

| Element | Value |
| --- | --- |
| State sₜ | 4 floats: cart position, cart velocity, pole angle, pole angular velocity |
| Actions | 2: push left (0), push right (1) |
| Reward | +1 for every step the pole stays up |
| Episode end | `terminated` (pole beyond ±12° or cart beyond ±2.4) or `truncated` (500-step limit) |
| Best possible return | 500 |

In [ ]:
# !pip install torch gymnasium numpy matplotlib   # uncomment if the packages are missing
import os

os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')

import copy, random, time
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import matplotlib.pyplot as plt

print("torch", torch.__version__, "| gymnasium", gym.__version__)

In [ ]:
# Take a look at the environment before you start
env = gym.make("CartPole-v1")
s, _ = env.reset(seed=0)
print("first state:", s)
print("observation space:", env.observation_space)
print("action space:", env.action_space)
env.close()

## Part 1 — Replay buffer

Implement uniform random experience replay as described in the slides:

- The buffer has **N** preallocated slots and a counter **c** of all transitions ever added.
- `add`: write at index $i = c \bmod N$ (this overwrites the oldest transition once the buffer is full), then increase $c$.
- `sample(n)`: only when $c \ge n$. Draw $n$ indices uniformly from $[0, \min(c, N))$.

Store each field in its own NumPy array (states, actions, rewards, next states, done flags). Then one mini-batch is a single fancy-indexing call such as `self.s[idx]`.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity, state_dim):
        self.N = capacity
        self.c = 0                                   # transitions ever added
        self.s    = np.zeros((capacity, state_dim), dtype=np.float32)
        self.a    = np.zeros(capacity, dtype=np.int64)
        self.r    = np.zeros(capacity, dtype=np.float32)
        self.s2   = np.zeros((capacity, state_dim), dtype=np.float32)
        self.done = np.zeros(capacity, dtype=np.float32)

    def add(self, s, a, r, s2, done):
        # TODO 1.1  compute i = c mod N  
        # TODO 1.2  write the five fields at index i (this overwrites the oldest)
        # TODO 1.3  increase c
        raise NotImplementedError

    def __len__(self):
        # TODO 1.4  number of valid transitions = min(c, N)
        raise NotImplementedError

    def sample(self, n):
        # TODO 1.5  return None while c < n
        # TODO 1.6  draw n random indices in [0, min(c, N)) and
        #           return (s, a, r, s2, done) as NumPy arrays
        raise NotImplementedError

**Self-check — Part 1.** This replays the slide example (N = 8, transitions d0 … d11).

In [ ]:
buf = ReplayBuffer(capacity=8, state_dim=4)
assert buf.sample(4) is None, "must refuse to sample before c >= n"
for k in range(11):                                  # add d0 ... d10
    buf.add(np.full(4, k), k % 2, float(k), np.full(4, k + 1), 0.0)
assert len(buf) == 8 and buf.c == 11
assert buf.r[3] == 3.0                               # slot 3 still holds d3
buf.add(np.full(4, 11), 1, 11.0, np.full(4, 12), 1.0)  # d11
assert buf.r[3] == 11.0, "d11 must overwrite slot 11 mod 8 = 3"
batches = [buf.sample(8) for _ in range(200)]
r  = np.concatenate([b[2] for b in batches])
s  = np.concatenate([b[0] for b in batches])
s2 = np.concatenate([b[3] for b in batches])
assert set(r.tolist()) == set(range(4, 12)), "samples must cover the whole buffer"
assert np.all(s2[:, 0] == s[:, 0] + 1), "fields of one transition must stay aligned"
print("ReplayBuffer: all checks passed")

### Questions — Part 1

**Q1.1** Why does `sample` draw from `[0, min(c, N))` and not from `[0, N)`?

**Your answer:**

_Write your answer here._

**Q1.2** A Python list of tuples with `pop(0)` would also work. Why is the ring buffer preferred when N = 50,000?

**Your answer:**

_Write your answer here._

**Q1.3** The buffer stores `done`, but the slides' tuple is (s, a, r, s′). Where in the algorithm is `done` needed?

**Your answer:**

_Write your answer here._

## Part 2 — Q-network

The Q-network is a small MLP. It takes a state and returns **one Q-value per action**, so a single forward pass scores every action.

Architecture: 4 inputs → 128 → 128 → 2 outputs, with ReLU between layers and **no activation on the output** (Q-values can be any real number).

You also write `select_action`, the ε-greedy policy: with probability ε pick a random action, otherwise pick $\arg\max_a \hat Q(s, a; \theta)$.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden=128):
        super().__init__()
        # TODO 2.1  nn.Sequential: Linear(state_dim, hidden) → ReLU →
        #           Linear(hidden, hidden) → ReLU → Linear(hidden, n_actions)
        self.net = None
        raise NotImplementedError

    def forward(self, x):
        return self.net(x)          # shape: (batch, n_actions)


def select_action(q_net, s, eps, n_actions):
    # TODO 2.2  with probability eps return a random action,
    #           otherwise return argmax_a Q(s, a; θ) as a Python int.
    #           Use torch.no_grad(). Hint: the network expects a batch,
    #           so convert s to a tensor of shape (1, state_dim).
    raise NotImplementedError

**Self-check — Part 2**

In [ ]:
q = QNetwork(4, 2)
out = q(torch.zeros(32, 4))
assert out.shape == (32, 2)
assert sum(p.numel() for p in q.parameters()) == 4*128+128 + 128*128+128 + 128*2+2
assert select_action(q, np.zeros(4, dtype=np.float32), 0.0, 2) in (0, 1)
print("QNetwork: all checks passed")

### Questions — Part 2

**Q2.1** Why does the network output Q(s, ·) for all actions, instead of taking (s, a) as input and returning one number?

**Your answer:**

_Write your answer here._

**Q2.2** Why must the last layer have no ReLU?

**Your answer:**

_Write your answer here._

## Part 3 — Target network and loss

The target network θ⁻ is a deep copy of θ that receives **no gradients**. The loss pulls $\hat Q(s_j, a_j; \theta)$ toward targets computed with θ⁻:

$$
\delta_j = \begin{cases} r_j & \text{if } s_{j+1} \text{ is terminal} \\ r_j + \gamma \max_{a'} \hat{Q}(s_{j+1}, a'; \theta^-) & \text{otherwise} \end{cases}
\qquad
l = \frac{1}{b}\sum_j \big(\delta_j - \hat{Q}(s_j, a_j; \theta)\big)^2
$$

Both cases fit in one line with a mask: $\delta = r + \gamma \cdot (1 - \text{done}) \cdot \max_{a'} \hat Q(s', a'; \theta^-)$.

(The slides use a sum; we use the mean, which only rescales the learning rate.)

> **Terminated ≠ truncated.** Gymnasium returns two flags. Store `done = terminated` only. When the 500-step time limit truncates an episode, the pole has *not* fallen, so the next state still has value and the target must bootstrap. Treating truncation as terminal is a classic bug: it teaches the agent that states near the time limit are worth nothing.

In [ ]:
def compute_loss(q_net, target_net, batch, gamma):
    s, a, r, s2, done = (torch.as_tensor(x) for x in batch)
    # TODO 3.1  Q(s_j, a_j; θ): run q_net(s) and pick the taken action
    #           with .gather(1, a.unsqueeze(1)).squeeze(1)
    # TODO 3.2  inside torch.no_grad(): max_a' Q(s_{j+1}, a'; θ⁻) from target_net(s2)
    # TODO 3.3  delta = r + gamma * (1 - done) * max_next
    # TODO 3.4  return the mean squared error between q_sa and delta
    raise NotImplementedError

**Self-check — Part 3**

In [ ]:
torch.manual_seed(0)
q = QNetwork(4, 2)
qt = copy.deepcopy(q)
b = 16
batch = (np.random.randn(b, 4).astype(np.float32), np.random.randint(0, 2, b),
         np.ones(b, np.float32), np.random.randn(b, 4).astype(np.float32),
         np.ones(b, np.float32))                       # all terminal
loss = compute_loss(q, qt, batch, gamma=0.99)
q_sa = q(torch.as_tensor(batch[0])).gather(1, torch.as_tensor(batch[1]).unsqueeze(1)).squeeze(1)
assert torch.isclose(loss, ((q_sa - 1.0) ** 2).mean()), "terminal target must be r"
loss.backward()
assert all(p.grad is None for p in qt.parameters()), "no gradient may reach θ⁻"
print("compute_loss: all checks passed")

### Questions — Part 3

**Q3.1** What would go wrong if you forgot `torch.no_grad()` in TODO 3.2 and the optimiser held both networks' parameters?

**Your answer:**

_Write your answer here._

**Q3.2** Why is `copy.deepcopy` needed? What happens with `target_net = q_net`?

**Your answer:**

_Write your answer here._

## Part 4 — Training loop

The loop is the DQN algorithm from the slides, line by line. The comments give the pseudocode line numbers from the reading, so you can check each step.

**Starting hyperparameters** — sensible defaults for CartPole, not tuned for this lab. Adjust them if learning is unstable.

| Symbol | Argument | Value |
| --- | --- | --- |
| K | `steps` | 50,000 |
| N | `capacity` | 50,000 |
| b | `batch` | 64 |
| m | `warmup` | 1,000 |
| C | `C` | 500 |
| γ | `gamma` | 0.99 |
| α | `lr` (Adam) | 0.001 |
| ε | `eps_end`, `eps_decay` | 1.0 → 0.05 over 10,000 steps |

`train` returns a log: a list of `(step, episode_return)` pairs, one per finished episode.

In [ ]:
def epsilon_by_step(i, eps_start, eps_end, eps_decay):
    # TODO 4.1  linear decay: eps_start at i = 0, eps_end from i = eps_decay onwards
    raise NotImplementedError


def train(steps=50_000, capacity=50_000, batch=64, warmup=1_000, C=500,
          gamma=0.99, lr=1e-3, eps_end=0.05, eps_decay=10_000,
          use_target=True, seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    env = gym.make("CartPole-v1")
    env.action_space.seed(seed)
    state_dim, n_actions = env.observation_space.shape[0], env.action_space.n
    q_net = QNetwork(state_dim, n_actions)
    target_net = copy.deepcopy(q_net)                      # θ⁻ ← θ
    opt = torch.optim.Adam(q_net.parameters(), lr=lr)
    buf = ReplayBuffer(capacity, state_dim)

    s, _ = env.reset(seed=seed)                            # line 1
    ep_ret, log = 0.0, []                                  # log: (step, return)
    for i in range(1, steps + 1):                          # line 2
        # TODO 4.2  eps = epsilon_by_step(i, 1.0, eps_end, eps_decay)       (line 6)
        # TODO 4.3  a = select_action(...); step the env:                     (lines 3–5)
        #           s2, r, terminated, truncated, _ = env.step(a)
        # TODO 4.4  buf.add(s, a, r, s2, float(terminated)); s ← s2;         (lines 7–8)
        #           add r to ep_ret
        # TODO 4.5  if terminated or truncated: append (i, ep_ret) to log,
        #           reset ep_ret to 0 and reset the env
        raise NotImplementedError

        if len(buf) > warmup and len(buf) > batch:         # line 9
            # TODO 4.6  sample a mini-batch; compute_loss with target_net
            #           (or q_net when use_target=False); then
            #           opt.zero_grad(), loss.backward(), opt.step()           (lines 10–15)
            # TODO 4.7  every C steps: target_net.load_state_dict(q_net.state_dict())
            raise NotImplementedError
    env.close()
    return log

**Self-check — Part 4**

1. Measure a random-policy baseline first: run with `eps_end=1.0` (the network is never used to act).
2. Train the full DQN. It should clearly beat the baseline within 50,000 steps and, on most seeds, reach episodes near the 500 cap.
3. If returns stay at the baseline, check that `opt.step()` runs, that the prediction is **not** computed with `target_net`, and that `done` is `terminated` only.

In [ ]:
assert abs(epsilon_by_step(0, 1.0, 0.05, 10_000) - 1.0) < 1e-9
assert abs(epsilon_by_step(5_000, 1.0, 0.05, 10_000) - 0.525) < 1e-9
assert abs(epsilon_by_step(20_000, 1.0, 0.05, 10_000) - 0.05) < 1e-9

t0 = time.time()
base_log = train(steps=20_000, eps_end=1.0, seed=0)
baseline = np.mean([ret for _, ret in base_log])
print(f"random baseline: mean return {baseline:.1f} over {len(base_log)} episodes ({time.time()-t0:.0f}s)")

In [ ]:
t0 = time.time()
log = train(seed=0)
steps_, rets_ = map(np.array, zip(*log))
print(f"{len(log)} episodes in {time.time()-t0:.0f}s | last 20 episodes: {rets_[-20:].mean():.1f} | best: {rets_.max():.0f}")

plt.figure(figsize=(8, 4))
plt.plot(steps_, rets_, alpha=0.3, label="episode return")
w = 20
plt.plot(steps_[w-1:], np.convolve(rets_, np.ones(w)/w, mode="valid"), label=f"moving average ({w} ep.)")
plt.axhline(baseline, ls="--", c="gray", label="random baseline")
plt.axhline(500, ls=":", c="k", label="max return (500)")
plt.xlabel("environment steps"); plt.ylabel("return"); plt.title("Full DQN on CartPole-v1 (seed 0)")
plt.legend(); plt.tight_layout(); plt.show()

## Part 5 — Experiments

Run each configuration with **3 seeds** (0, 1, 2). Plot the mean episode return ± one standard deviation against environment steps. Change **one thing at a time** from the full DQN of Part 4.

| Run | Change from default | Question it answers |
| --- | --- | --- |
| A — Full DQN | none | Baseline for every comparison |
| B — No target network | `use_target=False` | What does θ⁻ buy in stability? |
| C — Tiny buffer ≈ no replay | `capacity=128`, `warmup=64` | What happens when batches are nearly consecutive? |
| D — Sync period | `C` ∈ {1, 100, 500, 2000} | Where is the stability / staleness trade-off? |
| E — Buffer size | `capacity` ∈ {1,000; 10,000; 50,000} | How much memory is enough? |

The plotting helpers are given. Each configuration trains 3 times, so these cells take a while; lower `STEPS` for a quick first try, then run the full 50,000 steps for your report.

In [ ]:
STEPS = 50_000
SEEDS = (0, 1, 2)

def curve(log, grid, window=20):
    steps, rets = map(np.array, zip(*log))
    smooth = np.convolve(rets, np.ones(window) / window, mode="valid")
    return np.interp(grid, steps[window - 1:], smooth)

def run_configs(runs, steps=STEPS):
    grid = np.linspace(5_000, steps, 200)
    results = {}
    for label, kwargs in runs.items():
        t0 = time.time()
        results[label] = np.stack([curve(train(steps=steps, seed=sd, **kwargs), grid) for sd in SEEDS])
        print(f"{label:>16}: final {results[label][:, -1].mean():6.1f} ± {results[label][:, -1].std():5.1f}"
              f"   ({time.time()-t0:.0f}s)")
    return grid, results

def plot(grid, results, title, path):
    plt.figure(figsize=(8, 4.5))
    for label, ys in results.items():
        m, sd = ys.mean(0), ys.std(0)
        plt.plot(grid, m, label=label)
        plt.fill_between(grid, m - sd, m + sd, alpha=0.2)
    plt.axhline(baseline, ls="--", c="gray", lw=1, label="random baseline")
    plt.xlabel("environment steps"); plt.ylabel("episode return (moving avg, 20 ep.)")
    plt.title(title); plt.legend(); plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

**Runs A–C** — ablation of replay and the target network. Saves `ablation.png`.

In [ ]:
grid, res_abl = run_configs({
    "A full DQN":    {},
    "B no target":   {"use_target": False},
    "C tiny buffer": {"capacity": 128, "warmup": 64},
})
plot(grid, res_abl, "Ablation: replay and target network", "ablation.png")

**Run D** — target sync period C. Saves `sync_period.png`.

In [ ]:
# TODO 5.1  run C ∈ {1, 100, 500, 2000} with run_configs and plot the result
#           as "sync_period.png". Hint: a dict comprehension builds the runs.
raise NotImplementedError

**Run E** — replay capacity N. Saves `buffer_size.png`.

In [ ]:
# TODO 5.2  run capacity ∈ {1_000, 10_000, 50_000} and plot the result as "buffer_size.png"
raise NotImplementedError

### Questions — Part 5

**Q5.1** Rank runs A, B and C by final return and by variance across seeds. Link each gap to a problem from the slides (correlation, data efficiency, moving target).

**Your answer:**

_Write your answer here._

**Q5.2** In run D, which C learns fastest and which is most stable? Explain using the staircase picture of θ⁻.

**Your answer:**

_Write your answer here._

**Q5.3** Does `C = 1` behave like run B? Why or why not?

**Your answer:**

_Write your answer here._

**Q5.4** Why can a very large buffer slow learning early on, even though it decorrelates samples better?

**Your answer:**

_Write your answer here._

## Deliverables and grading

Submit one zip per pair with this notebook (all cells run, outputs visible), the figures and a report of at most 4 pages.

- [ ] Parts 1–3 pass their self-checks
- [ ] Full DQN learning curve with the random baseline marked
- [ ] `ablation.png` (runs A–C), `sync_period.png` (run D) and `buffer_size.png` (run E)
- [ ] Every question answered; each figure discussed in 3–5 sentences

| Criterion | Points |
| --- | --- |
| Replay buffer correct (ring indexing, sampling range, alignment) | 15 |
| Q-network and ε-greedy action selection | 10 |
| Loss: target network, `no_grad`, terminal masking, terminated vs truncated | 20 |
| Training loop matches the algorithm (warm-up, sync every C) | 15 |
| Experiments: 3 seeds, mean ± std plots, clear labels | 20 |
| Report: answers link results to correlation, data efficiency and moving targets | 20 |
| **Total** | **100** |

## References

- M. Hu, *The Art of Reinforcement Learning: Fundamentals, Mathematics, and Implementations with Python*, Apress 2023 — [code repository](https://github.com/Apress/art-of-reinforcement-learning)
- I. Gridin, *Practical Deep Reinforcement Learning with Python*, BPB Publications 2022 — [code repository](https://github.com/bpbpublications/Practical-Deep-Reinforcement-Learning-with-Python)
- [Building Multilayer Perceptron Models in PyTorch](https://machinelearningmastery.com/building-multilayer-perceptron-models-in-pytorch/), Machine Learning Mastery
- [Gymnasium CartPole-v1 documentation](https://gymnasium.farama.org/environments/classic_control/cart_pole/)